# Diagnóstico Inicial de Calidad de Datos con Python y Pandas

* **Denominación del Programa de Formación:** Tecnólogo en Análisis y Desarrollo de Software (Código: `228118`)
* **Ficha:** `3223873`
* **Competencia:** `220501094` - Estructurar propuesta técnica de servicio de tecnología de la información según requisitos técnicos y normativa
* **Resultado de Aprendizaje:** `220501094-01` - Definir especificaciones técnicas del software de acuerdo con las características del software a construir
* **Actividad de Aprendizaje:** `GA1-220501094-01-AA1-EV06` – Diagnosticar la calidad de datos: preparación, exploración y documentación
* **Aprendiz:** Simón Carmona Toro
* **Instructor:** Carlos Enrique Navia Torres

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1. Nombre y propósito del proyecto de ciencia de datos
* **Nombre del proyecto:** Ciencia de Datos Aplicada al Balance de Campeones, Tasas de Selección/Bloqueo (*Pick/Ban Rate*) y Escalado Estadístico por Clases en *League of Legends*.
* **Propósito del proyecto:** Analizar objetivamente mediante Python y Pandas el rendimiento competitivo de los campeones de *League of Legends* según su clase principal (Asesino, Luchador, Mago, Tirador, Tanque y Soporte), su índice de dificultad mecánica (del 1 al 10) y la duración de las partidas, con el fin de identificar campeones desbalanceados (*Overpowered* / *Underpowered*) y fundamentar un sistema de recomendación por carril.

### 2. Pregunta analítica formulada (recuperada de EV02 y EV03)
> *«¿De qué manera la clase de rol de un campeón (Asesino, Mago, Tirador, Tanque, Luchador o Soporte), su índice de dificultad mecánica y la duración total del encuentro influyen en la variación de su tasa de victoria (Win Rate), tasa de selección (Pick Rate) y tasa de bloqueo (Ban Rate) en el metajuego clasificatorio de League of Legends?»*

### 3. Datasets seleccionados para desarrollar el proyecto
1. **`lol_champion_meta_stats.csv`** *(Fuente: Kaggle Datasets – `vivovinco/league-of-legends-champion-stats`)*.
2. **`lol_champions_base_attributes.csv`** *(Fuente: Riot Games Data Dragon & Kaggle – `gyejr95/league-of-legends-champion-stats`)*.
3. **`lol_ranked_matches_performance.csv`** *(Fuente: Google Dataset Search & Kaggle – `paololol/league-of-legends-ranked-matches`)*.

### 4. Información que contiene cada archivo y su relación con el problema
* **`lol_champion_meta_stats.csv`:** Contiene las métricas del metajuego por campeón y carril (`Name`, `Class`, `Role`, `Tier`, `Score`, `Trend`, `Win %`, `Role %`, `Pick %`, `Ban %`, `KDA`). Aporta las variables dependientes centrales (`Win %`, `Pick %`, `Ban %`) para medir el éxito y el nivel de bloqueo de cada campeón en cada posición del mapa (`TOP`, `JUNGLE`, `MID`, `ADC`, `SUPPORT`).
* **`lol_champions_base_attributes.csv`:** Almacena el catálogo oficial de parámetros de diseño de fábrica de los campeones (`id`, `name`, `title`, `tags`, `partype`, `difficulty`, `hp`, `hpperlevel`, `mp`, `mpperlevel`, `armor`, `armorperlevel`, `attackdamage`, `attackdamageperlevel`). Permite incorporar el índice oficial de dificultad mecánica (`difficulty` de 1 a 10) y las tasas de crecimiento de estadísticas por nivel para explicar el escalado hacia el juego tardío.
* **`lol_ranked_matches_performance.csv`:** Registra el desempeño individual por partida clasificatoria (`match_id`, `champion_name`, `role`, `win`, `kills`, `deaths`, `assists`, `damage_dealt`, `gold_earned`, `duration_sec`, `patch`). Aporta la duración exacta de cada encuentro en segundos (`duration_sec`) y el daño infligido (`damage_dealt`), indispensables para evaluar cómo varía el rendimiento en partidas cortas frente a partidas largas.

### 5. Aspectos de calidad que se necesitan revisar antes de utilizar los datos
Antes de realizar cruces relacionales (`merge`), cálculos estadísticos o visualizaciones, es necesario revisar:
1. **Completitud (valores faltantes):** Verificar qué columnas presentan valores nulos (`NaN`) mediante `.info()`.
2. **Coherencia de tipos de datos (`dtypes`):** Comprobar si las variables porcentuales (`Win %`, `Pick %`, `Ban %`) están almacenadas como texto (`object`) debido al símbolo `%` en lugar de números decimales (`float64`).
3. **Consistencia de llaves de unión:** Revisar si los nombres de campeones con caracteres especiales o espacios (`Kai'Sa` vs. `Kaisa`, `Kha'Zix` vs. `Khazix`) están escritos de forma idéntica entre los tres archivos.
4. **Validez de rangos y valores atípicos:** Examinar mediante `.describe()` si existen duraciones de partida anormalmente bajas (`duration_sec < 300`, correspondientes a partidas canceladas o *remakes*) o inconsistencias entre `0` y `NaN` en estadísticas de maná (`mp`).

---
## Momento 2. Preparación del entorno, repositorio y carga de datos

Se importa la librería **Pandas** y se cargan los tres archivos CSV ubicados en `../data/raw/` en DataFrames independientes con nombres representativos (`df_meta`, `df_attributes` y `df_matches`).

In [1]:
import pandas as pd

df_meta = pd.read_csv("../data/raw/lol_champion_meta_stats.csv")
df_attributes = pd.read_csv("../data/raw/lol_champions_base_attributes.csv")
df_matches = pd.read_csv("../data/raw/lol_ranked_matches_performance.csv")

---
## Momento 3. Inspección de la estructura de los datasets

Para cada uno de los tres DataFrames se ejecutan los comandos solicitados en la guía: `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')`.

---
### 3.1. Inspección del Dataset 1: `df_meta` (`lol_champion_meta_stats.csv`)

In [2]:
df_meta.head()

      Name     Class     Role Tier  Score  Trend   Win %  Role % Pick %  Ban %   KDA
0   Aatrox   Fighter      TOP   S+  42.50  -4.50  46.80%  72.00%  0.65%  0.40%  1.85
1     Ahri      Mage      MID    S  58.65  -0.65  47.97%  73.05%  2.74%  2.70%  1.98
2    Akali  Assassin      MID    A  74.80   3.20  49.14%  74.10%  4.83%  5.00%  2.11
3  Alistar      Tank  SUPPORT    B  90.95    NaN  50.31%  75.15%  6.92%  7.30%  2.25
4    Amumu      Tank   JUNGLE    C  57.70   0.45  51.48%  76.20%  9.01%  9.60%  2.38

In [3]:
df_meta.tail()

        Name     Class     Role Tier  Score  Trend   Win %  Role % Pick %   Ban %   KDA
95      Jhin  Marksman   JUNGLE    D  45.35  -4.50  50.85%  29.00%  1.20%  28.90%  2.87
96      Jinx  Marksman  SUPPORT   S+  61.50  -0.65  52.02%  30.10%  3.29%  31.20%  3.00
97    Kai'Sa  Marksman      MID    S  77.65   3.20  53.19%  31.20%  5.38%  33.50%  3.13
98     Karma      Mage      TOP    A  44.40  -3.40  54.36%  32.30%  7.47%  35.80%  3.27
99  Katarina  Assassin      ADC    B  60.55   0.45  47.88%  33.40%  9.56%  38.10%  3.40

In [4]:
df_meta.shape

(100, 11)

In [5]:
df_meta.columns

Index(['Name', 'Class', 'Role', 'Tier', 'Score', 'Trend', 'Win %', 'Role %', 'Pick %', 'Ban %', 'KDA'], dtype='object')

In [6]:
df_meta.dtypes

Name       object
Class      object
Role       object
Tier       object
Score     float64
Trend     float64
Win %      object
Role %     object
Pick %     object
Ban %      object
KDA       float64
dtype: object

In [7]:
df_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    100 non-null    object 
 1   Class   96 non-null     object 
 2   Role    100 non-null    object 
 3   Tier    97 non-null     object 
 4   Score   100 non-null    float64
 5   Trend   93 non-null     float64
 6   Win %   100 non-null    object 
 7   Role %  100 non-null    object 
 8   Pick %  100 non-null    object 
 9   Ban %   100 non-null    object 
 10  KDA     100 non-null    float64
dtypes: float64(3), object(8)
memory usage: 8.7+ KB


In [8]:
df_meta.describe()

            Score      Trend         KDA
count  100.000000  93.000000  100.000000
mean    66.839000   0.432258    2.796800
std     14.343275   3.052001    0.552067
min     42.500000  -4.500000    1.850000
25%     54.612500  -2.300000    2.330000
50%     66.725000   0.450000    2.790000
75%     79.550000   3.200000    3.270000
max     90.950000   5.400000    3.760000

In [9]:
df_meta.describe(include='object')

          Name    Class Role Tier   Win %  Role % Pick %  Ban %
count      100       96  100   97     100     100    100    100
unique      70        6    5    6      85      45    100    100
top     Aatrox  Fighter  MID    A  46.80%  72.00%  0.65%  0.40%
freq         2       26   23   17       2       3      1      1

**Explicación de las características principales de `df_meta`:**
* Cuenta con **100 registros** y **11 columnas** (`3` de tipo `float64` y `8` de tipo `object`).
* El número de filas (`100`) es mayor al número de campeones únicos (`70` en `Name`) porque un mismo campeón puede desempeñarse en más de un carril (`Role`), registrando una fila por cada combinación campeón-carril.
* Se detectan valores faltantes en tres variables: `Class` (`96 non-null`, 4 ausentes), `Tier` (`97 non-null`, 3 ausentes) y `Trend` (`93 non-null`, 7 ausentes).
* Las variables porcentuales clave (`Win %`, `Role %`, `Pick %`, `Ban %`) aparecen como tipo `object` en lugar de `float64` debido a que incluyen el carácter `%` al final de cada valor.

---
### 3.2. Inspección del Dataset 2: `df_attributes` (`lol_champions_base_attributes.csv`)

In [10]:
df_attributes.head()

        id     name                title      tags     partype  difficulty   hp  hpperlevel     mp  mpperlevel  armor  armorperlevel  attackdamage  attackdamageperlevel
0   Aatrox   Aatrox     the Darkin Blade   Fighter  Blood Well           4  650         114    0.0         0.0     38           4.45            60                  5.00
1     Ahri     Ahri  the Nine-Tailed Fox      Mage        Mana           5  590          96  418.0        25.0     21           4.70            53                  3.00
2    Akali    Akali   the Rogue Assassin  Assassin      Energy           7  570         119  200.0         0.0     23           4.70            62                  3.30
3  Alistar  Alistar         the Minotaur      Tank        Mana           7  685         120  350.0        40.0     47           4.70            62                  3.75
4    Amumu    Amumu        the Sad Mummy      Tank        Mana           3  685          94  285.0        40.0     33           4.00            53         

In [11]:
df_attributes.tail()

        id    name                   title      tags partype  difficulty   hp  hpperlevel     mp  mpperlevel  armor  armorperlevel  attackdamage  attackdamageperlevel
65    Yone    Yone         the Unforgotten  Assassin    Flow           8  620         105    0.0         0.0     30            4.6            60                   2.0
66     Zed     Zed   the Master of Shadows  Assassin  Energy           7  654          99  200.0         0.0     32            4.7            63                   3.4
67   Ziggs   Ziggs  the Hexplosives Expert      Mage    Mana           4  606         106  480.0        23.5     22            4.5            54                   3.1
68  Zilean  Zilean        the Chronokeeper   Support    Mana           6  574          96  452.0        50.0     24            5.0            52                   3.0
69    Zyra    Zyra      Rise of the Thorns      Mage    Mana           7  574          93  418.0        25.0     29            4.2            53                   3.

In [12]:
df_attributes.shape

(70, 14)

In [13]:
df_attributes.columns

Index(['id', 'name', 'title', 'tags', 'partype', 'difficulty', 'hp', 'hpperlevel', 'mp', 'mpperlevel', 'armor', 'armorperlevel', 'attackdamage', 'attackdamageperlevel'], dtype='object')

In [14]:
df_attributes.dtypes

id                       object
name                     object
title                    object
tags                     object
partype                  object
difficulty                int64
hp                        int64
hpperlevel                int64
mp                      float64
mpperlevel              float64
armor                     int64
armorperlevel           float64
attackdamage              int64
attackdamageperlevel    float64
dtype: object

In [15]:
df_attributes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    70 non-null     object 
 1   name                  70 non-null     object 
 2   title                 70 non-null     object 
 3   tags                  70 non-null     object 
 4   partype               70 non-null     object 
 5   difficulty            70 non-null     int64  
 6   hp                    70 non-null     int64  
 7   hpperlevel            70 non-null     int64  
 8   mp                    65 non-null     float64
 9   mpperlevel            65 non-null     float64
 10  armor                 70 non-null     int64  
 11  armorperlevel         70 non-null     float64
 12  attackdamage          70 non-null     int64  
 13  attackdamageperlevel  70 non-null     float64
dtypes: float64(4), int64(5), object(5)
memory usage: 7.8+ KB


In [16]:
df_attributes.describe()

       difficulty          hp  hpperlevel          mp  mpperlevel      armor  armorperlevel  attackdamage  attackdamageperlevel
count   70.000000   70.000000   70.000000   65.000000   65.000000  70.000000      70.000000     70.000000             70.000000
mean     5.442857  622.314286  104.042857  322.646154   39.861538  30.571429       4.656429     58.571429              3.218571
std      2.004601   38.144858    8.523515  111.364047   18.427753   6.614809       0.670492      6.039895              0.755951
min      1.000000  530.000000   88.000000    0.000000    0.000000  19.000000       0.000000     40.000000              0.000000
25%      4.000000  591.250000   99.000000  280.000000   32.000000  26.000000       4.500000     54.000000              2.962500
50%      5.500000  630.000000  103.500000  339.000000   40.000000  30.000000       4.700000     60.000000              3.200000
75%      7.000000  650.000000  109.000000  400.000000   50.000000  34.000000       5.000000     62.00000

In [17]:
df_attributes.describe(include='object')

            id    name                 title     tags partype
count       70      70                    70       70      70
unique      70      70                    70        6       7
top     Aatrox  Aatrox  Daughter of the Void  Fighter    Mana
freq         1       1                     1       19      58

**Explicación de las características principales de `df_attributes`:**
* Contiene **70 registros únicos de campeones** y **14 columnas** (`5` de tipo `int64`, `4` de tipo `float64` y `5` de tipo `object`).
* El índice de dificultad (`difficulty`) se encuentra completo (`70 non-null`), variando entre `1` y `10` con un promedio de `5.44` y mediana de `5.50`.
* Las columnas `mp` y `mpperlevel` presentan **5 valores ausentes** (`65 non-null` de `70` registros) correspondientes a campeones sin recurso de maná (`partype == 'None'` o `'Shield'`/`'Flow'`), mientras que otros campeones sin maná tienen registrado `0.0`.
* Se observa una diferencia entre `id` (sin espacios ni apóstrofos, ej. `Kaisa`, `Chogath`, `LeeSin`) y `name` (con formato de visualización, ej. `Kai'Sa`, `Cho'Gath`, `Lee Sin`).

---
### 3.3. Inspección del Dataset 3: `df_matches` (`lol_ranked_matches_performance.csv`)

In [18]:
df_matches.head()

   match_id champion_name     role  win  kills  deaths  assists  damage_dealt  gold_earned  duration_sec  patch
0   4810001       Alistar      ADC    1      2       1        4       17960.0        11560          1320  14.19
1   4810001        Darius      TOP    0      5       6       11       22606.0        13438          1357  14.19
2   4810002          Jhin      TOP    1      8      11       18       27252.0        15316          1394  14.19
3   4810002      Malphite      MID    0     11       4        3       31898.0        14774          1431  14.19
4   4810003         Riven      MID    1     14       9       10       36544.0        16652          1468  14.19

In [19]:
df_matches.tail()

     match_id champion_name     role  win  kills  deaths  assists  damage_dealt  gold_earned  duration_sec  patch
495   4810248        Veigar  SUPPORT    0     15       4       15       42210.0        19630          1995  14.20
496   4810249          Zyra  SUPPORT    1      2       9        0       23656.0        13968          2032  14.20
497   4810249         Braum   JUNGLE    0      5       2        7       28302.0        15846          2069  14.20
498   4810250       Hecarim   JUNGLE    1      8       7       14       32948.0        17724          2106  14.20
499   4810250       Lee Sin      ADC    0     11       0       21       37594.0        19602          2143  14.20

In [20]:
df_matches.shape

(500, 11)

In [21]:
df_matches.columns

Index(['match_id', 'champion_name', 'role', 'win', 'kills', 'deaths', 'assists', 'damage_dealt', 'gold_earned', 'duration_sec', 'patch'], dtype='object')

In [22]:
df_matches.dtypes

match_id           int64
champion_name     object
role              object
win                int64
kills              int64
deaths             int64
assists            int64
damage_dealt     float64
gold_earned        int64
duration_sec       int64
patch            float64
dtype: object

In [23]:
df_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   match_id       500 non-null    int64  
 1   champion_name  500 non-null    object 
 2   role           491 non-null    object 
 3   win            500 non-null    int64  
 4   kills          500 non-null    int64  
 5   deaths         500 non-null    int64  
 6   assists        500 non-null    int64  
 7   damage_dealt   493 non-null    float64
 8   gold_earned    500 non-null    int64  
 9   duration_sec   500 non-null    int64  
 10  patch          500 non-null    float64
dtypes: float64(2), int64(7), object(2)
memory usage: 43.1+ KB


In [24]:
df_matches.describe()

           match_id         win       kills      deaths     assists  damage_dealt   gold_earned  duration_sec       patch
count  5.000000e+02  500.000000  500.000000  500.000000  500.000000    493.000000    500.000000    500.000000  500.000000
mean   4.810126e+06    0.500000    7.492000    5.492000   10.476000  30803.217039  16485.408000   1933.902000   14.195000
std    7.224048e+01    0.500501    4.606558    3.459751    6.353752   7299.638224   2208.675382    385.691608    0.005005
min    4.810001e+06    0.000000    0.000000    0.000000    0.000000  15460.000000  10788.000000    213.000000   14.190000
25%    4.810063e+06    0.000000    3.750000    2.000000    5.000000  24970.000000  14935.500000   1628.500000   14.190000
50%    4.810126e+06    0.500000    7.500000    5.500000   10.000000  30742.000000  16469.000000   1936.000000   14.195000
75%    4.810188e+06    1.000000   11.000000    8.250000   16.000000  36566.000000  18048.500000   2256.500000   14.200000
max    4.810250e+06    1

In [25]:
df_matches.describe(include='object')

       champion_name role
count            500  491
unique            70    5
top             Zyra  MID
freq               8  100

**Explicación de las características principales de `df_matches`:**
* Consta de **500 registros de desempeño en partida** y **11 columnas** (`7` de tipo `int64`, `2` de tipo `float64` y `2` de tipo `object`).
* Se identifican valores ausentes en `role` (`491 non-null`, **9 valores ausentes**) y en `damage_dealt` (`493 non-null`, **7 valores ausentes**, lo que provocó que Pandas asignara `float64` en lugar de `int64`).
* La estadística `describe()` sobre `duration_sec` muestra un promedio de `1.933,90` segundos (~32,2 minutos), pero revela un valor mínimo atípico de **`213` segundos** (~3,55 minutos), correspondiente a partidas canceladas prematuramente por desconexión (*remake* `< 300` s).

---
### 3.4. Tabla de diagnóstico estructural

| Dataset | Filas | Columnas | Tipos de datos | Observaciones |
| :--- | :---: | :---: | :--- | :--- |
| **Dataset 1:** `lol_champion_meta_stats.csv` (`df_meta`) | 100 | 11 | `float64` (3), `object` (8) | Presenta valores faltantes en `Class` (4), `Tier` (3) y `Trend` (7). Las columnas `Win %`, `Role %`, `Pick %` y `Ban %` están tipificadas como `object` por incluir el símbolo `%`. |
| **Dataset 2:** `lol_champions_base_attributes.csv` (`df_attributes`) | 70 | 14 | `int64` (5), `float64` (4), `object` (5) | Presenta 5 valores faltantes en `mp` y `mpperlevel` (`65 non-null`). Existe diferencia de formato entre `id` (`Kaisa`, `Chogath`) y `name` (`Kai'Sa`, `Cho'Gath`). |
| **Dataset 3:** `lol_ranked_matches_performance.csv` (`df_matches`) | 500 | 11 | `int64` (7), `float64` (2), `object` (2) | Presenta valores faltantes en `role` (9) y `damage_dealt` (7). Incluye duraciones de partida atípicas (`min = 213` s, partidas con *remake*) y `patch` como `float64`. |

---
## Momento 4. Diagnóstico inicial de calidad

A partir de los resultados obtenidos en el Momento 3, se examinan las cinco dimensiones de calidad solicitadas en la guía:

### 1. Valores faltantes
* **`df_meta` (`lol_champion_meta_stats.csv`):**
  * `Class` presenta **4 valores ausentes** (`96 non-null` de `100`): impediría agrupar esos 4 registros al calcular promedios por clase de campeón.
  * `Tier` presenta **3 valores ausentes** (`97 non-null` de `100`): afectaría los conteos por categoría de nivel competitivo (`S+`, `S`, `A`, `B`, `C`, `D`).
  * `Trend` presenta **7 valores ausentes** (`93 non-null` de `100`): reduce la muestra al analizar la variación entre parches.
* **`df_attributes` (`lol_champions_base_attributes.csv`):**
  * `mp` y `mpperlevel` presentan **5 valores ausentes** cada una (`65 non-null` de `70`), correspondientes a campeones que no utilizan maná (`Garen`, `Katarina`, `Mordekaiser`, `Riven`, `Yasuo`). Si no se imputan con `0.0`, esos campeones serían excluidos en análisis multivariados que involucren recursos base.
* **`df_matches` (`lol_ranked_matches_performance.csv`):**
  * `role` presenta **9 valores ausentes** (`491 non-null` de `500`), lo que impide segmentar esas 9 partidas por carril.
  * `damage_dealt` presenta **7 valores ausentes** (`493 non-null` de `500`), lo que impide calcular el Daño por Minuto (`DPM`) en esos registros.

### 2. Tipos de datos
* En **`df_meta`**, las columnas `Win %`, `Role %`, `Pick %` y `Ban %` tienen tipo **`object` (texto)** porque contienen el carácter `'%'` (ej. `'46.80%'`). Esto impide ejecutar operaciones matemáticas como `.mean()`, `.std()` o correlaciones hasta que se retire el símbolo `'%'` y se conviertan a `float64`.
* En **`df_matches`**, la columna `patch` está almacenada como **`float64`** (`14.19`, `14.20`) cuando conceptualmente representa una etiqueta de versión de software (`object` / `string`), y `damage_dealt` pasó a `float64` debido a la presencia de 7 valores `NaN`.

### 3. Posibles inconsistencias
* **Doble criterio para campeones sin maná en `df_attributes`:** Algunos campeones sin maná tienen registrado `mp = 0.0` (`Aatrox`, `Sett`, `Viego`, `Yone`), mientras que otros tienen `NaN` (`Garen`, `Katarina`, `Mordekaiser`, `Riven`, `Yasuo`), lo que constituye una inconsistencia de codificación para el mismo fenómeno.
* **Diferencias en nombres de campeones entre tablas:** En `df_attributes`, la columna `id` omite espacios y apóstrofos (`Chogath`, `Kaisa`, `Khazix`, `LeeSin`), mientras que `name` sí los conserva (`Cho'Gath`, `Kai'Sa`, `Kha'Zix`, `Lee Sin`), y en `df_matches` existen registros con grafías sin apóstrofo, lo que provocaría pérdida de cruces al unir tablas con `pd.merge()` si no se normalizan previamente.
* **Partidas canceladas (*remakes*) en `df_matches`:** El valor mínimo de `duration_sec` es `213` segundos (3,55 minutos). En *League of Legends*, las partidas de menos de 300 segundos (5 minutos) son partidas anuladas por desconexión (*remakes*), no partidas competitivas reales.

### 4. Limitaciones de los datos
* **Múltiples filas por campeón en `df_meta`:** Como `df_meta` tiene 100 filas para 70 campeones (debido a campeones jugados en más de un carril), hacer un `merge` directo con `df_attributes` sin considerar la columna `Role` duplicaría los atributos base de los campeones flexibles.
* **División por cero en cálculo de KDA por partida en `df_matches`:** La columna `deaths` tiene un mínimo de `0` (`min = 0.00`), por lo que calcular `(kills + assists) / deaths` directamente generaría valores infinitos (`inf`) si no se reemplaza `0` por `1` en el denominador.

### 5. Acciones de limpieza propuestas
1. **Conversión de porcentajes en `df_meta`:** Eliminar el carácter `'%'` con `.str.replace('%', '')` y convertir `Win %`, `Role %`, `Pick %` y `Ban %` a `float64`.
2. **Imputación de valores faltantes:**
   * En `df_meta`: Rellenar los 4 valores ausentes de `Class` consultando la columna `tags` de `df_attributes` mediante el nombre del campeón; imputar `Tier` con la moda (`'A'`) y `Trend` con `0.0`.
   * En `df_attributes`: Imputar los 5 valores ausentes de `mp` y `mpperlevel` con `0.0` para unificar todos los campeones sin maná.
   * En `df_matches`: Imputar los 9 valores faltantes de `role` con la posición principal del campeón y los 7 faltantes de `damage_dealt` con la mediana del campeón/rol.
3. **Estandarización de nombres (llaves primarias/foráneas):** Crear una llave normalizada en minúsculas y sin apóstrofos ni espacios para cruzar sin pérdidas `df_meta`, `df_attributes` y `df_matches`.
4. **Filtrado de partidas *remake*:** Filtrar en `df_matches` únicamente las partidas con `duration_sec >= 300` antes de analizar el escalado temporal.

---
### 4.1. Tabla resumen del diagnóstico inicial de calidad

| Dataset | Problema identificado | Evidencia encontrada | Posible impacto | Acción propuesta |
| :--- | :--- | :--- | :--- | :--- |
| **Dataset 1:** `lol_champion_meta_stats.csv` | Variables porcentuales almacenadas como texto (`object`). | `dtypes` e `info()` reportan `Win %`, `Role %`, `Pick %` y `Ban %` como `object` (ej. `'46.80%'`). | Impide calcular promedios, desviaciones estándar, filtros numéricos y correlaciones en Pandas. | Eliminar el símbolo `'%'` mediante `.str.replace('%', '')` y convertir las 4 columnas a tipo `float64`. |
| **Dataset 1:** `lol_champion_meta_stats.csv` | Valores faltantes en `Class` (4), `Tier` (3) y `Trend` (7). | `info()` reporta `96 non-null` en `Class`, `97 non-null` en `Tier` y `93 non-null` en `Trend` de `100` filas. | Exclusión de campeones al agrupar por clase (`Class`) o nivel (`Tier`). | Cruzar `Name` con `df_attributes['tags']` para recuperar `Class`; imputar `Tier` con la moda y `Trend` con `0.0`. |
| **Dataset 2:** `lol_champions_base_attributes.csv` | Valores faltantes e inconsistencia en `mp` y `mpperlevel` (`NaN` vs `0.0`). | `info()` muestra `65 non-null` de `70` en `mp` y `mpperlevel`, mientras `describe()` muestra `min = 0.0`. | Tratamiento desigual de campeones sin maná (`None`, `Flow`, `Shield`) en análisis estadísticos. | Imputar con `.fillna(0.0)` las columnas `mp` y `mpperlevel` cuando `partype` no sea `'Mana'`. |
| **Dataset 2 y 3:** `df_attributes` y `df_matches` | Diferencias de escritura en nombres de campeones (`id` vs `name` vs `champion_name`). | Presencia de apóstrofos y espacios en unos registros (`Kai'Sa`, `Cho'Gath`) y ausencia en otros (`Kaisa`, `Chogath`). | Pérdida de registros al unir tablas con `pd.merge()` si las cadenas no coinciden exactamente. | Estandarizar los nombres eliminando apóstrofos y espacios (`.str.replace("'", "").str.strip()`) antes del cruce. |
| **Dataset 3:** `lol_ranked_matches_performance.csv` | Valores faltantes en `role` (9) y `damage_dealt` (7), y partidas *remake* (`duration_sec < 300`). | `info()` reporta `491 non-null` en `role` y `493 non-null` en `damage_dealt`; `describe()` muestra `min = 213` s en `duration_sec`. | Sesgo hacia abajo en la duración promedio y daño por minuto por partidas canceladas al minuto 3. | Imputar `role` por moda del campeón, `damage_dealt` por mediana, convertir `patch` a `str` y filtrar `duration_sec >= 300`. |

---
## Momento 5. Conclusiones del diagnóstico para continuar con el proyecto

1. **Viabilidad técnica de los datasets:** Los tres conjuntos de datos seleccionados contienen la totalidad de las variables necesarias para responder la pregunta analítica del proyecto (clase de rol, índice de dificultad mecánica del 1 al 10, duración de la partida en segundos, daño infligido y tasas de victoria, selección y bloqueo).
2. **Prioridad de preparación de tipos de datos:** El hallazgo más crítico para la siguiente fase del proyecto es la conversión de `Win %`, `Role %`, `Pick %` y `Ban %` de tipo `object` a `float64` en `df_meta`, ya que son las métricas centrales del estudio.
3. **Control de calidad relacional y estadístico:** La proporción de datos faltantes es baja (entre el 1,4 % y el 7,1 % en las columnas afectadas) y completamente recuperable mediante imputación cruzada entre tablas, mientras que la detección temprana de partidas con `duration_sec < 300` evitará distorsiones al analizar el escalado hacia el juego tardío (*Late Game*).

*(El informe técnico detallado se encuentra documentado en [`reports/informe_diagnostico.md`](../reports/informe_diagnostico.md)).*